<a href="https://colab.research.google.com/github/ojaspaul123/DL-journey/blob/main/Functional-API/age_gender_revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [2]:
!kaggle datasets download -d jangedoo/utkface-new

Dataset URL: https://www.kaggle.com/datasets/jangedoo/utkface-new
License(s): copyright-authors
100% 331M/331M [00:02<00:00, 163MB/s]



In [3]:
import zipfile
zip = zipfile.ZipFile("/content/utkface-new.zip",'r')
zip.extractall("/content")
zip.close()

In [4]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [5]:
folder_path = '/content/utkface_aligned_cropped/UTKFace'

In [6]:
age=[]
gender=[]
img_path=[]
for file in os.listdir(folder_path):
  age.append(int(file.split('_')[0]))
  gender.append(int(file.split('_')[1]))
  img_path.append(file)

In [7]:
len(age)

23708

In [8]:
df = pd.DataFrame({'age':age,'gender':gender,'img':img_path})

In [9]:
df.shape

(23708, 3)

In [10]:
df.head()

,age,gender,img
0,52,0,52_0_0_20170111195813450.jpg.chip.jpg
1,53,0,53_0_0_20170109012810348.jpg.chip.jpg
2,35,0,35_0_1_20170116011302449.jpg.chip.jpg
3,27,1,27_1_4_20170113011320472.jpg.chip.jpg
4,42,1,42_1_2_20170116161958086.jpg.chip.jpg


In [11]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [12]:
train_df.shape

(20000, 3)

In [13]:
test_df.shape

(3708, 3)

In [14]:
train_datagen = ImageDataGenerator(rescale=1./255,
                                   rotation_range=30,
                                   width_shift_range=0.2,
                                   height_shift_range=0.2,
                                   shear_range=0.2,
                                   zoom_range=0.2,
                                   horizontal_flip=True)

test_datagen = ImageDataGenerator(rescale=1./255)

In [15]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                    class_mode='multi_output')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=folder_path,
                                                    x_col='img',
                                                    y_col=['age','gender'],
                                                    target_size=(200,200),
                                                  class_mode='multi_output')

Found 20000 validated image filenames.
Found 3708 validated image filenames.


In [16]:
import tensorflow as tf

def wrap_generator(gen):
    for x, y in gen:
        yield x, (y[0], y[1])  # convert list -> tuple

output_signature = (
    tf.TensorSpec(shape=(None, 200, 200, 3), dtype=tf.float32),
    (
        tf.TensorSpec(shape=(None,), dtype=tf.float32),  # age
        tf.TensorSpec(shape=(None,), dtype=tf.float32),  # gender
    )
)

train_ds = tf.data.Dataset.from_generator(
    lambda: wrap_generator(train_generator),
    output_signature=output_signature
)

test_ds = tf.data.Dataset.from_generator(
    lambda: wrap_generator(test_generator),
    output_signature=output_signature
)

In [17]:
from keras.applications.resnet50 import ResNet50
from keras.layers import *
from keras.models import Model

In [18]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [19]:
resnet = ResNet50(include_top=False, input_shape=(200,200,3))

resnet.trainable=False

output = resnet.layers[-1].output

flatten = Flatten()(output)

dense1 = Dense(512, activation='relu')(flatten)
dense2 = Dense(512,activation='relu')(flatten)

dense3 = Dense(512,activation='relu')(dense1)
dense4 = Dense(512,activation='relu')(dense2)

output1 = Dense(1,activation='linear',name='age')(dense3)
output2 = Dense(1,activation='sigmoid',name='gender')(dense4)

In [20]:
model = Model(inputs=resnet.input,outputs=[output1,output2])

In [21]:
model.compile(optimizer='adam', loss={'age': 'mae', 'gender': 'binary_crossentropy'}, metrics={'age': 'mae', 'gender': 'accuracy'},loss_weights={'age':1,'gender':99})

In [22]:
history = model.fit(
    train_ds,
    steps_per_epoch=len(train_generator),
    epochs=10,
    validation_data=test_ds,
    validation_steps=len(test_generator)
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 251s 372ms/step - age_loss: 15.4891 - age_mae: 15.4891 - gender_accuracy: 0.5114 - gender_loss: 0.8312 - loss: 97.7775 - val_age_loss: 14.8764 - val_age_mae: 14.8795 - val_gender_accuracy: 0.5297 - val_gender_loss: 0.6919 - val_loss: 83.3762
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 229s 366ms/step - age_loss: 14.9725 - age_mae: 14.9725 - gender_accuracy: 0.5210 - gender_loss: 0.6940 - loss: 83.6750 - val_age_loss: 14.9853 - val_age_mae: 14.9848 - val_gender_accuracy: 0.5305 - val_gender_loss: 0.6920 - val_loss: 83.4918
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 222s 355ms/step - age_loss: 14.8627 - age_mae: 14.8627 - gender_accuracy: 0.5215 - gender_loss: 0.6968 - loss: 83.8444 - val_age_loss: 14.4393 - val_age_mae: 14.4413 - val_gender_accuracy: 0.5259 - val_gender_loss: 0.6920 - val_loss: 82.9524
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 220s 352ms/step - age_loss: 14.8005 - age_mae: 14.8005 - gender_accuracy: 0.5214 - gender_loss: 0.6943 - loss: 83.